First prepare our files 

In [2]:
import pandas as pd
master=pd.read_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\master_orders.csv',parse_dates=['order_purchase_timestamp'])
rfm=pd.read_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\rfm_segments.csv')
churn=pd.read_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\churn_scores.csv')
seg_summary=pd.read_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\segment_summary.csv')

TABLEAU FILE 1:-MAIN ORDERS FACT TABLE

In [3]:
#Add a clean year-monthstring column for time series
master['year_month_str']=(
    master['order_purchase_timestamp'].dt.strftime('%Y-%m')
)
# keep only delivered orders for revenue analysis
master_delivered=master[master['order_status']=='delivered'].copy()
master_delivered.to_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\tableau_orders.csv',index=False)
print(f'tableau_orders.csv saved: {master_delivered.shape}')

tableau_orders.csv saved: (110197, 37)


TABLEAU FILE 2:-CUSTOMER SEGMENTS

In [6]:
rfm_tableau=rfm.merge(
    churn[['customer_unique_id','churn_probability','churn_risk_tier']],
    on='customer_unique_id',
    how='left'
)
rfm_tableau.to_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\tableau_rfm.csv',index=False)
print('tableau_rfm.csv saved successfully',rfm_tableau.shape)

tableau_rfm.csv saved successfully (95420, 13)


In [9]:
rfm_tableau.groupby('churn_risk_tier')['customer_unique_id'].count()

churn_risk_tier
Critical Risk    92987
High Risk           94
Low Risk          1966
Medium Risk        373
Name: customer_unique_id, dtype: int64

TABLEAU FILE 3:-MONTHLY REVENUE FOR TREND CHART

In [14]:
monthly=(
    master_delivered
    .groupby('year_month_str')
    .agg(
        total_revenue= ('revenue_per_item','sum'),
        total_orders= ('order_id','nunique'),
        avg_order_value=('revenue_per_item','mean')
    )
    .reset_index()
    .round(2)
)
monthly.to_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\tableau_monthly.csv',index=False)
print('tableau_monthly.csv saved successfully',monthly.shape)

tableau_monthly.csv saved successfully (23, 4)


TABLEAU FILE 4:- STATE LEVEL SUMMARY

In [ ]:
state_summary=(
    master_delivered.groupby('customer_state')
    .agg(
        total_revenue=('revenue_per_item','sum'),
        total_orders=('order_id','nunique'),
        avg_order_value=('revenue_per_item','mean'),
        late_delivery_pct=('is_late','mean'),
        avg_review_score=('review_score','mean')
    )
    .reset_index()
    .round(2)
)
state_summary['late_delivery_pct']=(
    state_summary['late_delivery_pct']*100
).round(1)
state_summary.to_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\tableau_states.csv',index=False)
print('tableau_states.csv saved succesfully',state_summary.shape)



tableau_states.csv saved succesfully (27, 6)


Fix this by setting the country context. We need to add a country field.

In [1]:
import pandas as pd
states=pd.read_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\tableau_states.csv')

In [2]:
# Add country column — Tableau uses this to resolve ambiguity
states['country'] = 'Brazil'

# Also add full state names — helps Tableau match correctly
state_name_map = {
    'SP': 'São Paulo',        'RJ': 'Rio de Janeiro',
    'MG': 'Minas Gerais',     'RS': 'Rio Grande do Sul',
    'PR': 'Paraná',           'SC': 'Santa Catarina',
    'BA': 'Bahia',            'DF': 'Distrito Federal',
    'GO': 'Goiás',            'ES': 'Espírito Santo',
    'PE': 'Pernambuco',       'CE': 'Ceará',
    'PA': 'Pará',             'MT': 'Mato Grosso',
    'MS': 'Mato Grosso do Sul','MA': 'Maranhão',
    'PB': 'Paraíba',          'RN': 'Rio Grande do Norte',
    'AL': 'Alagoas',          'PI': 'Piauí',
    'SE': 'Sergipe',          'RO': 'Rondônia',
    'AM': 'Amazonas',         'TO': 'Tocantins',
    'AC': 'Acre',             'AP': 'Amapá',
    'RR': 'Roraima'
}

states['state_name'] = states['customer_state'].map(state_name_map)
states.to_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\tableau_states.csv',index=False)
print("Updated tableau_states.csv:")
print(states.head())


Updated tableau_states.csv:
  customer_state  total_revenue  total_orders  avg_order_value  \
0             AC       19575.33            80           215.11   
1             AL       94172.49           397           220.54   
2             AM       27585.47           145           169.24   
3             AP       16141.81            67           199.28   
4             BA      591137.81          3256           160.50   

   late_delivery_pct  avg_review_score country state_name  
0                3.0              4.13  Brazil       Acre  
1               21.0              3.83  Brazil    Alagoas  
2                3.0              4.11  Brazil   Amazonas  
3                4.0              4.26  Brazil      Amapá  
4               12.0              3.86  Brazil      Bahia  


TABLEAU FILE 5:-CATEGORY PERFORMANCE

In [11]:
category=(
    master_delivered.groupby('product_category_name_english')
    .agg(
        total_revenue=('revenue_per_item','sum'),
        total_orders=('order_id','nunique'),
        avg_price=('price','mean'),
        avg_review=('review_score','mean')
    )
    .reset_index()
    .sort_values('total_revenue',ascending=False)
    .round(2)
)
category.to_csv(r'J:\python\DA_project\Olist_Ecommerce_intelligence\data\processed\tableau_categories.csv',index=False)
print('tableau_categories.csv saved successfully',category.shape)
print('\n All 5 Tableau files are Ready')

tableau_categories.csv saved successfully (71, 5)

 All 5 Tableau files are Ready
